# 🧩 Day 05: Advanced Linked List — The Interview Favorites

---

## The Problems That Actually Show Up

These three problems pop up in interviews more than almost any other linked list questions:

1. **LRU Cache** — Amazon, Google, Meta's favorite
2. **Copy List with Random Pointer** — tests deep understanding of references
3. **Add Two Numbers** — simple but tricky with carry handling

Each one teaches a different skill: LRU is about combining data structures, Copy Random is about cloning with complex pointers, and Add Two Numbers is about linked list arithmetic.

---

## 🎯 Problem 1: LRU Cache (LC #146)

Design a data structure that follows Least Recently Used (LRU) eviction policy.

```
Operations:
  get(key)        → return value if exists, else -1. Mark as recently used.
  put(key, value) → insert or update. If capacity exceeded, evict least recent.

Both operations must be O(1).
```

**Why is this hard?** You need:
- O(1) lookup → **HashMap** (dict)
- O(1) insertion/deletion by order of use → **Doubly Linked List**
- Track the order of access → items "move to front" when accessed

**The combo:** HashMap maps key → node in the doubly linked list. The list maintains access order. Most recently used is at the head, least recently used is at the tail.

```
╔══════════════════════════════════════════════════════════════╗
║  LRU Cache = HashMap + Doubly Linked List                   ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  HashMap:  key → node reference (O(1) lookup)               ║
║                                                              ║
║  DLL:  HEAD ⇔ [most recent] ⇔ ... ⇔ [least recent] ⇔ TAIL ║
║        (dummy)                                   (dummy)     ║
║                                                              ║
║  get(key): find node, move to front, return value            ║
║  put(key): update/create node, move to front                 ║
║            if over capacity → remove node at tail            ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
```

In [ ]:
class DLLNode:
    """Doubly linked list node for LRU Cache."""
    def __init__(self, key=0, val=0):
        self.key = key
        self.val = val
        self.prev = None
        self.next = None

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.cache = {}  # key → DLLNode
        
        # Dummy head and tail — eliminates edge cases
        self.head = DLLNode()  # Most recently used side
        self.tail = DLLNode()  # Least recently used side
        self.head.next = self.tail
        self.tail.prev = self.head
    
    def _remove(self, node):
        """Remove a node from the DLL."""
        node.prev.next = node.next
        node.next.prev = node.prev
    
    def _add_to_front(self, node):
        """Add node right after head (most recently used position)."""
        node.prev = self.head
        node.next = self.head.next
        self.head.next.prev = node
        self.head.next = node
    
    def get(self, key):
        if key in self.cache:
            node = self.cache[key]
            self._remove(node)       # Remove from current position
            self._add_to_front(node) # Move to front (most recent)
            return node.val
        return -1
    
    def put(self, key, value):
        if key in self.cache:
            self._remove(self.cache[key])  # Remove old position
        
        node = DLLNode(key, value)
        self.cache[key] = node
        self._add_to_front(node)  # Add as most recent
        
        if len(self.cache) > self.cap:
            # Evict least recently used (node just before tail)
            lru = self.tail.prev
            self._remove(lru)
            del self.cache[lru.key]  # This is why nodes store the key!

# Demo
lru = LRUCache(2)
lru.put(1, 1)
lru.put(2, 2)
print(lru.get(1))   # 1 (marks key 1 as recently used)
lru.put(3, 3)        # Evicts key 2 (least recently used)
print(lru.get(2))   # -1 (was evicted)
lru.put(4, 4)        # Evicts key 1
print(lru.get(1))   # -1 (was evicted)
print(lru.get(3))   # 3
print(lru.get(4))   # 4

**Why do nodes store the key?** When we evict the LRU node from the tail, we need to delete it from the hashmap too. We need the key for that. Since we only have a reference to the DLL node at the tail, the node itself must carry the key.

This is the #1 question people miss in the implementation.

---

## 🎯 Problem 2: Copy List with Random Pointer (LC #138)

Deep copy a linked list where each node has a `next` pointer AND a `random` pointer that can point to any node in the list (or None).

```
Input:  [7 → 13 → 11 → 10 → 1]  with random pointers:
        7.random = None
        13.random = 7
        11.random = 1
        10.random = 11
        1.random = 7
```

**The challenge:** When you create a copy of node A, and A.random points to node C, you need the copy of A to point to the **copy** of C — not the original C. But you might not have created the copy of C yet.

**Solution: HashMap** — map each original node to its clone. Two passes:
1. First pass: create all clones (map old → new)
2. Second pass: wire up next and random pointers using the map

In [ ]:
class RandomNode:
    def __init__(self, val=0, next=None, random=None):
        self.val = val
        self.next = next
        self.random = random

def copyRandomList(head):
    """
    Deep copy with random pointers using a hashmap.
    Time: O(n), Space: O(n)
    """
    if not head:
        return None
    
    # Map: original node → clone node
    old_to_new = {}
    
    # Pass 1: Create all clones
    curr = head
    while curr:
        old_to_new[curr] = RandomNode(curr.val)
        curr = curr.next
    
    # Pass 2: Wire up next and random
    curr = head
    while curr:
        clone = old_to_new[curr]
        clone.next = old_to_new.get(curr.next)      # Could be None
        clone.random = old_to_new.get(curr.random)   # Could be None
        curr = curr.next
    
    return old_to_new[head]

# Build the example
n1 = RandomNode(7)
n2 = RandomNode(13)
n3 = RandomNode(11)
n4 = RandomNode(10)
n5 = RandomNode(1)
n1.next = n2; n2.next = n3; n3.next = n4; n4.next = n5
n1.random = None; n2.random = n1; n3.random = n5
n4.random = n3; n5.random = n1

# Copy and verify
clone = copyRandomList(n1)
curr = clone
while curr:
    r_val = curr.random.val if curr.random else "None"
    print(f"  val={curr.val}, random→{r_val}")
    curr = curr.next

# Verify it's a TRUE deep copy (different objects)
print(f"\nOriginal and clone are different objects: {n1 is not clone}")

---

## 🎯 Problem 3: Add Two Numbers (LC #2)

Two numbers are represented as reversed linked lists. Add them.

```
Input:  (2 → 4 → 3) + (5 → 6 → 4)   ← represents 342 + 465
Output:  7 → 0 → 8                     ← represents 807
```

**The nice thing:** The digits are already in reverse order (units digit first), so we can add left-to-right — just like how you'd do long addition by hand! Just track the carry.

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def addTwoNumbers(l1, l2):
    """
    Add two numbers represented as reversed linked lists.
    Time: O(max(n, m)), Space: O(max(n, m))
    """
    dummy = ListNode(0)
    curr = dummy
    carry = 0
    
    while l1 or l2 or carry:
        # Get values (0 if list is exhausted)
        v1 = l1.val if l1 else 0
        v2 = l2.val if l2 else 0
        
        total = v1 + v2 + carry
        carry = total // 10
        digit = total % 10
        
        curr.next = ListNode(digit)
        curr = curr.next
        
        # Advance pointers
        l1 = l1.next if l1 else None
        l2 = l2.next if l2 else None
    
    return dummy.next

def build_list(values):
    dummy = ListNode(0)
    curr = dummy
    for v in values:
        curr.next = ListNode(v)
        curr = curr.next
    return dummy.next

def print_list(head):
    parts = []
    while head:
        parts.append(str(head.val))
        head = head.next
    print(" → ".join(parts))

# 342 + 465 = 807
l1 = build_list([2, 4, 3])  # 342
l2 = build_list([5, 6, 4])  # 465
result = addTwoNumbers(l1, l2)
print("342 + 465 =", end=" "); print_list(result)  # 7 → 0 → 8

# 99 + 1 = 100 (carry propagation)
l3 = build_list([9, 9])
l4 = build_list([1])
result2 = addTwoNumbers(l3, l4)
print("99 + 1   =", end=" "); print_list(result2)   # 0 → 0 → 1

The key to getting this right: the `while l1 or l2 or carry` condition. The `or carry` handles the case where both lists are done but there's a leftover carry (like 99 + 1 = 100).

---

## 🗺️ Advanced LL Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  ADVANCED LINKED LIST PATTERNS                                      ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. LRU CACHE: HashMap + Doubly Linked List                         ║
║     get/put O(1) — nodes store key so we can delete from hashmap    ║
║     Dummy head/tail eliminate edge cases                             ║
║                                                                      ║
║  2. COPY WITH RANDOM: HashMap old_node → clone_node                 ║
║     Pass 1: create clones. Pass 2: wire up next & random.           ║
║     Alternative: interleave clones (O(1) space but trickier)        ║
║                                                                      ║
║  3. ADD TWO NUMBERS: Digit-by-digit with carry                      ║
║     Don't forget: while l1 or l2 or carry (the 'or carry'!)        ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 146 | LRU Cache | Medium | HashMap + DLL |
| 460 | LFU Cache | Hard | HashMap + DLL + freq map |
| 138 | Copy List with Random Pointer | Medium | HashMap clone |
| 2 | Add Two Numbers | Medium | Digit addition + carry |
| 445 | Add Two Numbers II | Medium | Reverse + add (or stack) |
| 430 | Flatten Multilevel Doubly LL | Medium | DFS / stack |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 LRU Cache = dict + doubly linked list. Both O(1). Top asked.   │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Copy Random: hashmap old→new, two passes (create then wire)    │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Add Numbers: handle unequal lengths + final carry gracefully   │
├──────────────────────────────────────────────────────────────────────┤
│  📌 LRU nodes store KEY because eviction needs to delete from dict │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Advanced LL = combining LL with other data structures          │
└──────────────────────────────────────────────────────────────────────┘
```

---

## 🎉 Chapter 02 Complete — Linked Lists Mastered!

You've covered every major linked list pattern:
- Fundamentals (traversal, dummy head, reverse, merge, Floyd's)
- Two-pointer (middle, nth from end, cycle start, palindrome)
- Reversal in depth (partial, swap pairs, k-group)
- Merge & sort (merge sort, merge K, intersection)
- Advanced (LRU Cache, copy random, add numbers)

**Next chapter:** Stacks & Queues — where LIFO meets FIFO.